In [ ]:
from pyspark.sql import SparkSession

# criando sessão spark
spark = SparkSession.builder \
    .appName("Bank Transactions Analysis") \
    .getOrCreate()

In [ ]:
from pyspark.sql.functions import *

# lendo csv das transações
df = spark.read.csv(
    "../data/transactions.csv",
    header=True,
    inferSchema=True,
    sep=";"
)

# convertendo vírgula decimal pra ponto
df = df.withColumn(
    "value",
    regexp_replace("value", ",", ".").cast("double")
)

# mostrando os dados
df.show()

# vendo os tipos das colunas
df.printSchema()

In [ ]:
# removendo linhas com valores nulos
df = df.dropna()

# pegando só valores positivos
df = df.filter(col("value") > 0)

In [ ]:
# quantidade de transações por cliente
client_transactions = df.groupBy("client_id") \
    .count() \
    .orderBy(col("count").desc())

client_transactions.show()

In [ ]:
# total gasto por cliente
client_expenses = df.groupBy("client_id") \
    .agg(sum("value").alias("total_expenses")) \
    .orderBy(col("total_expenses").desc())

client_expenses.show()

In [ ]:
# média de gasto dos clientes
client_avg = df.groupBy("client_id") \
    .agg(avg("value").alias("avg_expenses"))

client_avg.show()

In [ ]:
# separando transações suspeitas
# nesse caso peguei valores acima de 5000
suspicious_transactions = df.filter(col("value") > 5000)

suspicious_transactions.show()

In [ ]:
# exportando resultado pra csv
suspicious_transactions.toPandas().to_csv(
    "../output/suspicious_transactions.csv",
    index=False
)